# 02.02 PyPTO Hello World 加法算子

Hello World 示例用于建立 PyPTO 算子开发的最小闭环。这个闭环包含四个动作：准备输入 Tensor，定义计算规则，执行计算，再用 PyTorch 结果进行验证。

本节的计算目标是逐元素加法：

```text
out = x + y
```

`x` 和 `y` 是形状相同的输入 Tensor，`out` 是输出 Tensor。逐元素加法表示两个输入在相同位置上的数字相加，例如 `[1, 2, 3] + [10, 20, 30]` 得到 `[11, 22, 33]`。PyPTO kernel 负责描述这个计算，PyTorch 负责构造输入数据和参考结果。


## 1. 输入输出规格

在实现算子之前，需要先明确输入、输出、shape 和 dtype。shape 表示 Tensor 每个维度的长度，dtype 表示每个元素的数据类型。

| 名称 | 含义 | shape | dtype |
| --- | --- | --- | --- |
| `input_data0` | 第一个输入 Tensor | `(1, 4, 1, 64)` | FP32 |
| `input_data1` | 第二个输入 Tensor | `(1, 4, 1, 64)` | FP32 |
| `output_data` | 加法结果输出 Tensor | `(1, 4, 1, 64)` | FP32 |

三个 Tensor 的 shape 保持一致，因为逐元素加法要求每个位置都能一一对应。`(1, 4, 1, 64)` 一共包含 `1 * 4 * 1 * 64 = 256` 个元素，输出 Tensor 的 256 个位置分别保存两个输入对应位置的和。


## 2. 完整代码

下面的代码单元给出完整的 Hello World 加法算子流程。代码中包含依赖导入、设备选择、PyPTO kernel 定义、输入输出构造、运行调用和误差验证。

运行模式分为 `npu` 和 `sim`。当 `torch_npu` 可用时，示例使用 NPU 模式；如果没有 NPU 扩展，则使用 SIM 模式。`TILE_FWK_DEVICE_ID` 用于指定进程内的 NPU 设备编号，未设置时默认使用 `0`。需要指定物理可见卡时，可以在代码开头设置 `PREFERRED_ASCEND_RT_VISIBLE_DEVICES`，并在修改后重启 kernel。


In [5]:
#!/usr/bin/env python3
# coding: utf-8
"""
Hello World Example for PyPTO

This notebook cell runs the full hello_world.py flow in one step.
"""

import os
import sys
import argparse
import pypto
import torch
import numpy as np
from numpy.testing import assert_allclose

# ---- 设备选择说明 ----
# PyPTO 教程使用两个环境变量控制 NPU 设备选择：
#
# 1. ASCEND_RT_VISIBLE_DEVICES：指定当前进程可见的物理 NPU 卡号。
#    必须在 import torch_npu 之前设置才生效。
#    设置后，可见的物理卡会被重新编号，逻辑编号从 0 开始。
#    例如 "13" 表示只让物理卡 13 可见，它在进程内变成逻辑 npu:0。
#    "5,8" 表示让物理卡 5 和 8 可见，它们分别变成逻辑 npu:0 和 npu:1。
#    保持 None 表示不限制，所有物理卡都可见。
#
# 2. TILE_FWK_DEVICE_ID：在可见卡中选择使用哪一张逻辑设备。
#    默认值为 "0"，即使用第一张可见卡。
#    如果可见卡不止一张，可以设置为 "1"、"2" 等选其他逻辑设备。
#
# 两者的关系：ASCEND_RT_VISIBLE_DEVICES 决定「哪些卡可见 + 映射规则」，
# TILE_FWK_DEVICE_ID 决定「在可见卡中选哪一张」。
# ASCEND_RT_VISIBLE_DEVICES = "13"

ASCEND_RT_VISIBLE_DEVICES = "13"
if ASCEND_RT_VISIBLE_DEVICES is not None:
    os.environ["ASCEND_RT_VISIBLE_DEVICES"] = str(ASCEND_RT_VISIBLE_DEVICES)

try:
    import torch_npu
except ImportError:
    torch_npu = None


def get_device_id():
    """
    Get and validate TILE_FWK_DEVICE_ID from environment variable.

    Returns:
        int: The device ID if valid, None otherwise.
    """
    if torch_npu is None:
        print("torch_npu is not available; NPU execution will be skipped.")
        return None

    try:
        return int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    except ValueError:
        print(f"ERROR: TILE_FWK_DEVICE_ID must be an integer, got: {os.environ['TILE_FWK_DEVICE_ID']}")
        return None


def create_add_kernel(shape: tuple, run_mode: str = "npu"):
    if run_mode == "npu":
        mode = pypto.RunMode.NPU
    elif run_mode == "sim":
        mode = pypto.RunMode.SIM
    else:
        raise ValueError(f"Invalid run_mode: {run_mode}. Must be 'npu' or 'sim'")

    @pypto.frontend.jit(runtime_options={"run_mode": mode})
    def add_kernel(
        x: pypto.Tensor([...], pypto.DT_FP32),
        y: pypto.Tensor([...], pypto.DT_FP32),
        out: pypto.Tensor([...], pypto.DT_FP32),
    ):
        pypto.set_vec_tile_shapes(1, 4, 1, 64)
        out[:] = x + y

    return add_kernel


def test_add_direct(device_id=None, run_mode: str = "npu") -> None:
    device = f"npu:{device_id}" if (run_mode == "npu" and device_id is not None) else "cpu"
    shape = (1, 4, 1, 64)

    print("Running on device:", device)
    print("Run mode:", run_mode)
    print("Input shape:", shape)

    input_data0 = torch.rand(shape, dtype=torch.float, device=device)
    input_data1 = torch.rand(shape, dtype=torch.float, device=device)
    output_data = torch.empty(shape, dtype=torch.float32, device=device)

    create_add_kernel(shape, run_mode)(input_data0, input_data1, output_data)

    golden = torch.add(input_data0, input_data1)
    max_diff = np.abs(output_data.cpu().numpy() - golden.cpu().numpy()).max()

    print(f"Input0 shape: {input_data0.shape}")
    print(f"Input1 shape: {input_data1.shape}")
    print(f"Output shape: {output_data.shape}")
    print(f"Max difference: {max_diff:.6f}")

    assert_allclose(output_data.cpu().numpy(), golden.cpu().numpy(), rtol=3e-3, atol=3e-3)
    print("✓ Hello world example passed")
    print()

device_id = get_device_id()
test_add_direct(device_id)

Running on device: npu:0
Run mode: npu
Input shape: (1, 4, 1, 64)
Input0 shape: torch.Size([1, 4, 1, 64])
Input1 shape: torch.Size([1, 4, 1, 64])
Output shape: torch.Size([1, 4, 1, 64])
Max difference: 0.000000
✓ Hello world example passed



## 3. 运行结果说明

成功执行时，输出会包含运行模式、设备、输入输出 shape 和最大误差。典型输出形式如下：

```text
Running on device: npu:0
Run mode: npu
Input shape: (1, 4, 1, 64)
Input0 shape: torch.Size([1, 4, 1, 64])
Input1 shape: torch.Size([1, 4, 1, 64])
Output shape: torch.Size([1, 4, 1, 64])
Max difference: 0.000000
✓ Hello world example passed
```

`Running on device: npu:0` 表示 Tensor 创建在当前进程可见的第 0 张 NPU 上。
`Max difference` 是 PyPTO 输出和 PyTorch 参考结果之间的最大差值；数值在误差阈值内时，`assert_allclose` 校验通过。
最后的通过信息表示加法算子的运行和验证闭环完成。


## 4. 源码结构

这个示例可以分为 Host 侧代码和 Kernel 侧代码。

| 层次 | 作用 | 对应函数 |
| --- | --- | --- |
| Host 侧 | 读取设备、创建 Tensor、调用 kernel、校验结果 | `get_device_id`、`test_add_direct`、`main` |
| Kernel 侧 | 描述要执行的 Tensor 计算 | `create_add_kernel` 内部的 `add_kernel` |

Host 侧代码由 Python 和 PyTorch 执行，负责把输入、输出和运行参数准备好。Kernel 侧代码由 `@pypto.frontend.jit` 标记，描述需要交给 PyPTO 编译和执行的计算。


## 5. 设备编号和设备字符串

`device_id` 是设备编号，`device` 是创建 Tensor 时使用的设备字符串。二者的关系如下：

```python
device_id = 0
device = "npu:0"
```

当运行模式是 `npu` 且 `device_id` 有效时，输入、输出 Tensor 创建在 `npu:0` 上；当运行模式是 `sim` 或没有可用 NPU 时，Tensor 创建在 `cpu` 上。

在设置可见设备列表的场景下，进程内部的 NPU 编号可能从 0 重新开始。例如某个物理设备被设置为当前进程唯一可见设备后，代码中仍然使用 `npu:0` 表示这张可见设备。切换物理卡需要在 `torch_npu` 初始化前完成，因此修改可见设备配置后需要重启 kernel。


## 6. `create_add_kernel`：定义计算规则

`create_add_kernel` 根据运行模式选择 PyPTO 的执行后端，并返回一个 JIT kernel。JIT 可以理解为“在运行时记录并编译计算描述”的机制。

```python
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def add_kernel(...):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    out[:] = x + y
```

`pypto.Tensor([...], pypto.DT_FP32)` 描述 kernel 的输入输出参数是 FP32 Tensor。这里的 `pypto.Tensor` 不是创建真实数据，而是描述进入 PyPTO kernel 的参数类型。真实数据由 Host 侧的 PyTorch 创建，再传入 PyPTO kernel。


## 7. Tile Shape 和输出写回

Kernel 内部的核心语句是：

```python
pypto.set_vec_tile_shapes(1, 4, 1, 64)
out[:] = x + y
```

Tile Shape 可以先理解为底层执行时使用的数据分块形状。它不改变数学结果，只影响计算如何组织。本例是向量类逐元素计算，因此使用 `set_vec_tile_shapes`。这里的 Tile Shape 与输入 shape 一致，表示按 `(1, 4, 1, 64)` 这一组维度组织计算。

`out[:] = x + y` 表示将加法结果写回输出 Tensor 的全部位置。`out` 是调用者提前分配好的输出缓冲区，kernel 不重新创建输出对象，而是把结果放入这个缓冲区。


## 8. `test_add_direct`：构造数据并执行 kernel

`test_add_direct` 将完整验证流程串起来。首先用 PyTorch 构造两个随机输入：

```python
input_data0 = torch.rand(shape, dtype=torch.float).to(device)
input_data1 = torch.rand(shape, dtype=torch.float).to(device)
```

然后分配输出：

```python
output_data = torch.empty(shape, dtype=torch.float32, device=device)
```

随机输入先在 CPU 上生成，再通过 `.to(device)` 搬到目标设备。这样可以避免依赖设备侧随机数算子，使示例重点集中在 PyPTO 加法 kernel 的执行和验证上。输出 Tensor 仍然直接在目标设备上分配，随后调用 `add_kernel(input_data0, input_data1, output_data)`，加法结果被写入 `output_data`。


## 9. PyTorch 参考结果和误差验证

验证闭环使用 PyTorch 生成参考结果：

```python
golden = torch.add(input_data0, input_data1)
```

`golden` 的数学含义与 kernel 中的 `out[:] = x + y` 一致，因此可以作为参考答案。随后将 PyPTO 输出和 PyTorch 输出转到 CPU 并转换为 NumPy 数组：

```python
output_np = output_data.cpu().numpy()
golden_np = golden.cpu().numpy()
```

误差验证使用：

```python
assert_allclose(output_np, golden_np, rtol=3e-3, atol=3e-3)
```

`rtol` 是相对误差阈值，`atol` 是绝对误差阈值。FP32 加法在这个示例中通常可以得到非常小的误差；保留阈值是为了形成通用的验证写法。


## 10. `main`：组织示例入口

`main` 负责组织示例入口，而不是描述算子计算本身。它完成三件事：

1. 使用 `argparse` 解析 `--run_mode`、`--list` 和示例 ID。
2. 使用 `examples` 字典登记可运行的示例函数。
3. 根据运行模式准备设备，并调用 `test_add_direct`。

这种组织方式适合后续扩展多个示例。新增示例时，可以在 `examples` 字典中增加新的 ID、名称、说明和函数入口，而不需要改变已有加法 kernel 的实现。


## 11. 补充说明

`torch` 版本中出现 `+cpu` 并不必然表示无法使用 NPU。NPU 能力由 `torch_npu` 扩展提供；当 `torch_npu` 可用时，PyTorch 可以通过扩展创建 NPU Tensor。

SIM 模式用于在模拟路径中验证代码结构和编译路径，NPU 模式用于在真实设备上执行 kernel。二者使用同一套 kernel 描述，区别在于 `pypto.RunMode` 的取值不同。

设备侧执行通常是异步的。如果设备执行发生异常，错误可能在后续 `.cpu()` 或同步操作中暴露。因此，验证阶段的同步和结果拷回也是检查 kernel 是否真正执行成功的重要步骤。


## 12. 拓展：修改计算表达式

将加法扩展为乘加形式时，kernel 中的计算表达式可以改为：

```python
out[:] = (x + y) * 2
```

对应的 PyTorch 参考结果也需要同步修改：

```python
golden = torch.add(input_data0, input_data1) * 2
```

验证仍然使用同一套 `assert_allclose` 逻辑。这个扩展说明，PyPTO kernel 的计算表达式和 PyTorch 参考实现需要保持同一个数学含义。


## 本节小结

Hello World 加法算子展示了 PyPTO 开发的基本闭环：明确输入输出规格，定义 JIT kernel，设置 Tile Shape，写回输出 Tensor，构造 PyTorch 参考结果，并完成误差验证。

在这个闭环中，`create_add_kernel` 描述计算规则，`test_add_direct` 负责运行和验证，`main` 负责组织示例入口。后续更复杂的算子仍然沿用这条主线，只是计算表达式、shape 关系和 Tile 设置会更加丰富。
